In [1]:
# Célula 1
!pip install vllm sentence-transformers

Autenticado com sucesso via Secrets!


In [2]:
from huggingface_hub import login
from google.colab import userdata

meu_token = userdata.get('HF_TOKEN')
login(token=meu_token)

print("Autenticado com sucesso via Secrets!")

Autenticado com sucesso via Secrets!


In [1]:
# Célula 3: Motor 7B (Forçando variáveis no Sistema Operacional)
%env VLLM_USE_V1=0
%env VLLM_WORKER_MULTIPROC_METHOD=spawn

from vllm import LLM, SamplingParams

print("Iniciando Qwen 7B na GPU A100 (BFloat16 + Deep Layers)...")

try:
    llm = LLM(
        model="Qwen/Qwen2.5-7B-Instruct",
        dtype="bfloat16",
        max_model_len=1024,
        gpu_memory_utilization=0.8,
        enforce_eager=False,
        distributed_executor_backend="uni"
    )
    print("\n✅ Motor de 7B carregado! VRAM alocada com sucesso.")
except Exception as e:
    print(f"\n❌ Falha: {e}")

env: VLLM_USE_V1=0
env: VLLM_WORKER_MULTIPROC_METHOD=spawn
Iniciando Qwen 7B na GPU A100 (BFloat16 + Deep Layers)...

✅ Motor de 7B carregado! VRAM alocada com sucesso.


In [5]:
# Célula 4 V5: Simulador com Captura de Telemetria (Latência e VRAM)
import time
import torch
import random
import numpy as np
import pandas as pd
from collections import Counter
from vllm import SamplingParams

sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=10, seed=42)
prompt_bruto = "Defina o conceito de Transformação Digital em exatamente 6 palavras, nem mais, nem menos."
prompt_formatado = llm.get_tokenizer().apply_chat_template([{"role": "user", "content": prompt_bruto}], tokenize=False, add_generation_prompt=True)

grupos_carga = [int(x) for x in np.linspace(1, 250, 30)]
iteracoes_por_grupo = 20
dados_brutos = []

for carga in grupos_carga:
    print(f"🚀 Testando patamar de {carga} requisições simultâneas...")
    for i in range(iteracoes_por_grupo):
        if carga > 1:
            prompts_ruido = [f"Explain tech in {random.randint(10, 500)} words."] * (carga - 1)
            lote = [prompt_formatado] + [
                llm.get_tokenizer().apply_chat_template([{"role":"user","content":p}], tokenize=False, add_generation_prompt=True)
                for p in prompts_ruido
            ]
        else:
            lote = [prompt_formatado]

        torch.cuda.reset_peak_memory_stats()
        start_time = time.time()
        out = llm.generate(lote, sampling_params, use_tqdm=False)
        end_time = time.time()
        latencia_segundos = end_time - start_time
        free_mem, total_mem = torch.cuda.mem_get_info()
        vram_gb = (total_mem - free_mem) / (1024 ** 3)

        texto_gerado = out[0].outputs[0].text.strip()
        dados_brutos.append({
            "Carga": carga,
            "Iteracao": i + 1,
            "Resposta": texto_gerado,
            "Latencia_Segundos": latencia_segundos,
            "Pico_VRAM_GB": vram_gb
        })
        torch.cuda.empty_cache()

df_bruto = pd.DataFrame(dados_brutos)
print("\n✅ Teste Concluído!")

✅ Teste Concluído!


In [6]:
# Célula 6 V5: Processamento de Telemetria e Drift Semântico
from sentence_transformers import SentenceTransformer, util
modelo_emb = SentenceTransformer('all-MiniLM-L6-v2')

resultados_agrupados = []
df_bruto['Similaridade_Cosseno'] = 100.0

for carga in df_bruto['Carga'].unique():
    subset = df_bruto[df_bruto['Carga'] == carga]
    contagem = Counter(subset['Resposta'])
    dominante = contagem.most_common(1)[0][0]
    ocorrencias_dominante = contagem.most_common(1)[0][1]
    taxa_erro = ((20 - ocorrencias_dominante) / 20) * 100
    emb_dominante = modelo_emb.encode(dominante, convert_to_tensor=True)

    for idx, row in subset.iterrows():
        if row['Resposta'] != dominante:
            emb_var = modelo_emb.encode(row['Resposta'], convert_to_tensor=True)
            sim = util.cos_sim(emb_dominante, emb_var).item() * 100
            df_bruto.at[idx, 'Similaridade_Cosseno'] = sim

    resultados_agrupados.append({
        "Carga": carga,
        "Taxa_Erro_%": taxa_erro,
        "Latencia_Media_s": subset['Latencia_Segundos'].mean(),
        "VRAM_Media_GB": subset['Pico_VRAM_GB'].mean(),
        "Similaridade_Media_%": subset['Similaridade_Cosseno'].mean()
    })

df_agrupado = pd.DataFrame(resultados_agrupados)
correlacao = df_agrupado['Carga'].corr(df_agrupado['Taxa_Erro_%'])
print(f"RELATÓRIO ESTATÍSTICO GERADO | Pearson: {correlacao:.4f}")

RELATÓRIO ESTATÍSTICO GERADO | Pearson: 0.2296


In [9]:
# Célula 7 V5: Painel de Múltiplos Resultados
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Análise Sistêmica de Estresse em Motor de Inferência (vLLM / A100)', fontsize=16, fontweight='bold', y=1.02)

sns.regplot(ax=axes[0, 0], data=df_agrupado, x='Carga', y='Taxa_Erro_%', order=3, ci=None, scatter_kws={'s': 50}, line_kws={'color': 'red', 'linestyle': '--'})
axes[0, 0].axvline(x=50, color='gray', linestyle=':', label='Ponto de Ruptura Estimado')
axes[0, 0].set_title('1. Quebra de Reprodutibilidade (%)', fontweight='bold')

sns.lineplot(ax=axes[0, 1], data=df_agrupado, x='Carga', y='Latencia_Media_s', color='purple', marker='o')
axes[0, 1].set_title('2. Degradação de Latência (Tempo de Resposta)', fontweight='bold')

axes[1, 0].fill_between(df_agrupado['Carga'], df_agrupado['VRAM_Media_GB'], color="skyblue", alpha=0.4)
sns.lineplot(ax=axes[1, 0], data=df_agrupado, x='Carga', y='VRAM_Media_GB', color='dodgerblue')
axes[1, 0].axhline(y=32, color='red', linestyle='--', alpha=0.6, label='Limite Físico Configurado (80%)')
axes[1, 0].set_title('3. Consumo Físico de Memória VRAM', fontweight='bold')

sns.boxplot(ax=axes[1, 1], data=df_bruto, x='Carga', y='Similaridade_Cosseno', color='lightgreen', showfliers=False)
for ind, label in enumerate(axes[1, 1].get_xticklabels()):
    if ind % 4 != 0: label.set_visible(False)
axes[1, 1].set_title('4. Dispersão do Drift Semântico', fontweight='bold')

plt.tight_layout()
plt.savefig('Painel_Resultados_TCC_Alta_Resolucao.png', dpi=300, bbox_inches='tight')
plt.show()
df_agrupado.to_excel('Tabela_Estatistica_Completa_TCC.xlsx', index=False)